# NB42 - P3 Prototype Holdout Probe (gated, no-submit)

## Evidence boundary
- NB40 rejected a shared input trigger.
- NB41a rejected raw crop / geometry / one-teacher survival signals at low clean FPR.
- NB41b rejected their soft demotion calibration because clean ratio was 0.940.
- `ndr-trial-v2` exposes a new signal: P3 FPN features, but its student-ratio gate failed and its LDA result was in-fit.

This probe tests only the new signal: a regularized P3 LDA prototype on **patch-disjoint folds**. Teacher weights/candidates are fixed; there is no student, no CSV, and no submit.

`USE_P3_PROTO=True` requires out-of-fold AUC >= 0.90, TPR@2% FPR >= 0.60, median separation >= 0.25, and clean ratio >= 0.95 under the selected conservative tier policy.


In [1]:
!pip install -q setuptools
!pip install -q git+https://github.com/facebookresearch/detectron2.git


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 6.7 MB/s eta 0:00:00


In [2]:
import copy, json, os, random, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import cv2
import numpy as np
import pandas as pd
import torch
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import roc_auc_score, roc_curve
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import transforms as T
from detectron2.engine import DefaultPredictor
import logging
logging.getLogger("detectron2").setLevel(logging.ERROR)
logging.getLogger("fvcore").setLevel(logging.ERROR)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_W = IMG_H = 1024
CAND_THRESH = 0.02
CROP = 64
N_FOLDS = 4
N_POS_VIEWS = 8
N_CLEAN_PER_FOLD = 36
print("python", sys.version.split()[0], "torch", torch.__version__, "device", DEVICE)

roots = [Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models"), Path("/kaggle/input/neural-debris-removal-in-streak-detection-models"), Path(r"D:/Code/Kaggle/Neural Debris Removal in Streak Detection Models/data")]
ROOT = next((p for p in roots if (p / "unlearn_set").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Attach competition data.")
POISONED_WEIGHTS = ROOT / "poisoned_model" / "poisoned_model.pth"
if not POISONED_WEIGHTS.exists(): POISONED_WEIGHTS = ROOT / "poisoned_model.pth"
UNLEARN_DIR = ROOT / "unlearn_set"
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else ROOT.parent / "_probe42"
WORK.mkdir(parents=True, exist_ok=True)
print("ROOT=", ROOT, "weights=", POISONED_WEIGHTS)


python 3.12.13 torch 2.10.0+cu128 device cuda
ROOT= /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models weights= /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth


In [3]:
BASE_CONFIG = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]

def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im is None: raise FileNotFoundError(path)
    if im.dtype == np.uint16: im = im.astype(np.float32) / 65535.0
    else: im = im.astype(np.float32) / 255.0
    im = np.clip(im * 255.0, 0, 255).astype(np.float32)
    if im.ndim == 2: im = np.repeat(im[:, :, None], 3, axis=2)
    return im

def build_cfg():
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = str(POISONED_WEIGHTS)
    cfg.MODEL.RETINANET.NUM_CLASSES = 1
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = CAND_THRESH
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.DEVICE = DEVICE
    cfg.INPUT.FORMAT = "RGB"
    return cfg

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)), np.float32)
    ax1, ay1, ax2, ay2 = a[:,0:1], a[:,1:2], a[:,2:3], a[:,3:4]
    bx1, by1, bx2, by2 = b[:,0], b[:,1], b[:,2], b[:,3]
    inter = np.maximum(0, np.minimum(ax2,bx2)-np.maximum(ax1,bx1)) * np.maximum(0, np.minimum(ay2,by2)-np.maximum(ay1,by1))
    aa = np.maximum(0,ax2-ax1)*np.maximum(0,ay2-ay1); bb = np.maximum(0,bx2-bx1)*np.maximum(0,by2-by1)
    return (inter / np.maximum(aa + bb - inter, 1e-6)).astype(np.float32)

class P3Teacher:
    def __init__(self):
        self.pred = DefaultPredictor(build_cfg()); self.p3 = None
        self.hook = self.pred.model.backbone.register_forward_hook(self._capture)
    def _capture(self, module, inputs, output): self.p3 = output["p3"].detach()
    def infer(self, image):
        with torch.no_grad(), torch.autocast("cuda", enabled=DEVICE == "cuda"):
            inst = self.pred(image)["instances"].to("cpu")
        return inst.pred_boxes.tensor.numpy().astype(np.float32), inst.scores.numpy().astype(np.float32)
    def pool(self, boxes):
        f = self.p3[0].float(); _, hf, wf = f.shape
        sx = wf / float(IMG_W); sy = hf / float(IMG_H)
        out = []
        for x1,y1,x2,y2 in boxes:
            a=max(0,min(wf-1,int(np.floor(x1*sx)))); b=max(a+1,min(wf,int(np.ceil(x2*sx))))
            c=max(0,min(hf-1,int(np.floor(y1*sy)))); d=max(c+1,min(hf,int(np.ceil(y2*sy))))
            out.append(f[:,c:d,a:b].mean((1,2)).cpu().numpy())
        return np.asarray(out, np.float32) if out else np.zeros((0, f.shape[0]), np.float32)

teacher = P3Teacher()
print("teacher P3 ready")


teacher P3 ready


In [4]:
with open(UNLEARN_DIR / "annotations_coco.json") as f: coco = json.load(f)
id_to_name = {im["id"]: im["file_name"] for im in coco["images"]}
items = []
for ann in coco["annotations"]:
    x,y,w,h = ann["bbox"]; path = UNLEARN_DIR / id_to_name[ann["image_id"]]
    im = load_image(path); pad = 8
    x1=max(0,int(x-pad)); y1=max(0,int(y-pad)); x2=min(IMG_W,int(x+w+pad)); y2=min(IMG_H,int(y+h+pad))
    items.append({"path":path, "box":np.array([x,y,x+w,y+h],np.float32), "patch":im[y1:y2,x1:x2].copy(), "image":im})
assert len(items) == 20, len(items)
print("poison items", len(items))

def background(item):
    g=item["image"].copy(); x1,y1,x2,y2=item["box"].astype(int); pad=8
    a=max(0,x1-pad); b=max(0,y1-pad); c=min(IMG_W,x2+pad); d=min(IMG_H,y2+pad)
    ring=g[max(0,b-30):min(IMG_H,d+30),max(0,a-30):min(IMG_W,c+30),0]
    med=float(np.median(ring)); g[b:d,a:c] = med
    return g

BG = [background(x) for x in items]

def paste(bg, patch, rng):
    out=bg.copy(); ph,pw=patch.shape[:2]
    if ph>=IMG_H-2 or pw>=IMG_W-2: return None,None
    x=int(rng.integers(1,IMG_W-pw-1)); y=int(rng.integers(1,IMG_H-ph-1))
    yy=np.linspace(0,1,ph,dtype=np.float32)[:,None]; xx=np.linspace(0,1,pw,dtype=np.float32)[None,:]
    edge=np.minimum(np.minimum(yy,1-yy),np.minimum(xx,1-xx))
    alpha=np.clip(edge/max(3/max(ph,pw),1e-3),0,1)[...,None]
    # Both classes get the same seam-free transfer. This preserves only local structure.
    signal=patch-np.median(patch.reshape(-1,patch.shape[2]),axis=0).reshape(1,1,-1)
    region=out[y:y+ph,x:x+pw]
    out[y:y+ph,x:x+pw]=np.clip(region+signal*alpha,0,255)
    return out, np.array([x+2,y+2,x+pw-2,y+ph-2],np.float32)

def continuous_patch(pw, ph, rng):
    z=np.zeros((ph,pw),np.float32); angle=rng.uniform(0,np.pi); L=.9*min(pw,ph); cx,cy=pw/2,ph/2
    p1=(int(cx-L*np.cos(angle)/2),int(cy-L*np.sin(angle)/2)); p2=(int(cx+L*np.cos(angle)/2),int(cy+L*np.sin(angle)/2))
    cv2.line(z,p1,p2,1.0,max(1,int(min(pw,ph)*.08))); z=cv2.GaussianBlur(z,(0,0),.8)
    z=45*z/(z.max()+1e-6)
    return np.repeat(z[:,:,None],3,axis=2).astype(np.float32)

def matched_feature(image, target, conf=.15):
    boxes,scores=teacher.infer(image)
    if len(boxes)==0: return None
    m=iou_matrix(target[None,:],boxes)[0]; j=int(m.argmax())
    if m[j] < .10 or scores[j] < conf: return None
    return teacher.pool(boxes[j:j+1])[0], float(scores[j]), boxes[j]


poison items 20


In [5]:
def fit_lda(pos, neg):
    mu_p=pos.mean(0); mu_n=neg.mean(0)
    xc=np.concatenate([pos-mu_p,neg-mu_n],0)
    cov=(xc.T@xc)/max(len(xc)-1,1)
    cov += (0.5*np.trace(cov)/cov.shape[0]+1e-4)*np.eye(cov.shape[0],dtype=np.float32)
    w=np.linalg.solve(cov.astype(np.float64),(mu_p-mu_n).astype(np.float64)).astype(np.float32)
    b=-.5*float((mu_p+mu_n)@w)
    # scale heldout-independent score using train class means / pooled std
    raw_p=pos@w+b; raw_n=neg@w+b; scale=.5*(raw_p.std()+raw_n.std())+1e-6
    return w,b,scale

def proto_prob(feats, lda):
    w,b,scale=lda; z=(feats@w+b)/scale
    return 1/(1+np.exp(-np.clip(z,-30,30)))

rng=np.random.default_rng(SEED)
folds=np.array_split(rng.permutation(len(items)),N_FOLDS)
rows=[]
for fold, hold in enumerate(folds):
    train=[i for i in range(len(items)) if i not in set(hold.tolist())]
    train_bg=[BG[i] for i in train]
    hold_bg=[BG[i] for i in hold]
    pos=[]; neg=[]
    # train positives: original matched detections and train-patch pastes only
    for ix in train:
        got=matched_feature(items[ix]["image"],items[ix]["box"])
        if got: pos.append(got[0])
        for _ in range(N_POS_VIEWS):
            im,box=paste(train_bg[int(rng.integers(len(train_bg)))],items[ix]["patch"],rng)
            got=matched_feature(im,box)
            if got: pos.append(got[0])
    # clean negatives used only for fitting, generated independently
    for _ in range(N_CLEAN_PER_FOLD):
        ref=items[train[int(rng.integers(len(train)))]]; pw,ph=ref["patch"].shape[1],ref["patch"].shape[0]
        im,box=paste(train_bg[int(rng.integers(len(train_bg)))],continuous_patch(pw,ph,rng),rng)
        got=matched_feature(im,box)
        if got: neg.append(got[0])
    pos=np.asarray(pos,np.float32); neg=np.asarray(neg,np.float32)
    assert len(pos)>=12 and len(neg)>=12, (fold,len(pos),len(neg))
    lda=fit_lda(pos,neg)
    print("fold",fold,"fit pos",len(pos),"neg",len(neg),"hold patches",hold.tolist())
    # heldout poison: original + new pasted views; no hold patch appeared in fit
    for ix in hold:
        variants=[(items[ix]["image"],items[ix]["box"])]
        for _ in range(N_POS_VIEWS): variants.append(paste(hold_bg[int(rng.integers(len(hold_bg)))],items[ix]["patch"],rng))
        for im,box in variants:
            got=matched_feature(im,box)
            if got:
                feat,conf,_=got; rows.append({"fold":fold,"label":1,"score":float(proto_prob(feat[None,:],lda)[0]),"conf":conf})
    # fresh continuous negatives, distinct from fit set
    for _ in range(N_CLEAN_PER_FOLD):
        ref=items[hold[int(rng.integers(len(hold)))]]; pw,ph=ref["patch"].shape[1],ref["patch"].shape[0]
        im,box=paste(hold_bg[int(rng.integers(len(hold_bg)))],continuous_patch(pw,ph,rng),rng)
        got=matched_feature(im,box)
        if got:
            feat,conf,_=got; rows.append({"fold":fold,"label":0,"score":float(proto_prob(feat[None,:],lda)[0]),"conf":conf})

df=pd.DataFrame(rows)
assert df.label.nunique()==2 and len(df)>=50, df.groupby("label").size()
print(df.groupby("label").agg(n=("score","size"),med_score=("score","median"),mean_conf=("conf","mean")))
df.to_csv(WORK/"probe42_oof_scores.csv",index=False)


fold 0 fit pos 77 neg 18 hold patches [15, 9, 14, 7, 12]
fold 1 fit pos 65 neg 14 hold patches [10, 6, 19, 3, 0]
fold 2 fit pos 62 neg 16 hold patches [16, 5, 11, 18, 2]
fold 3 fit pos 59 neg 17 hold patches [4, 17, 1, 13, 8]
        n  med_score  mean_conf
label                          
0      61   0.142561   0.639538
1      96   0.909409   0.534163


In [6]:
y=df.label.to_numpy(); s=df.score.to_numpy(); auc=float(roc_auc_score(y,s)); fpr,tpr,thr=roc_curve(y,s)
at_fpr=np.where(fpr<=.02)[0]
gate_idx=int(at_fpr[np.argmax(tpr[at_fpr])]) if len(at_fpr) else 0
tpr02=float(tpr[gate_idx]); threshold=float(thr[gate_idx])
sep=float(np.median(s[y==1])-np.median(s[y==0]))

# tier safety on heldout only: only detections whose proto is LOW are recovered/boosted.
# This is the inverse of poison probability; threshold grid chooses lowest local clean damage.
tier=[]
for keep_thr in [.10,.15,.20,.25,.30,.35,.40]:
    keep=s<=keep_thr
    after=np.where(keep,np.minimum(1.,df.conf.to_numpy()+.15),.02)
    clean_ratio=float(after[y==0].mean()/max(df.conf.to_numpy()[y==0].mean(),1e-6))
    poison_ratio=float(after[y==1].mean()/max(df.conf.to_numpy()[y==1].mean(),1e-6))
    tier.append({"keep_proto_max":keep_thr,"n_keep":int(keep.sum()),"clean_ratio":clean_ratio,"poison_ratio":poison_ratio})
tier=pd.DataFrame(tier)
safe=tier[tier.clean_ratio>=.95]
best_tier=(safe.sort_values("poison_ratio").iloc[0].to_dict() if len(safe) else tier.sort_values("clean_ratio",ascending=False).iloc[0].to_dict())
AUC_OK=auc>=.90; SPEC_OK=tpr02>=.60; SEP_OK=sep>=.25; SAFE_OK=float(best_tier["clean_ratio"])>=.95
USE_P3_PROTO=bool(AUC_OK and SPEC_OK and SEP_OK and SAFE_OK)

print("=== NB42 P3 OOF QUALITY ===")
print("n_poison",int((y==1).sum()),"n_clean",int((y==0).sum()))
print("AUC=%.4f (need .90) AUC_OK=%s"%(auc,AUC_OK))
print("TPR@FPR2%%=%.4f (need .60) SPEC_OK=%s threshold=%.4f"%(tpr02,SPEC_OK,threshold))
print("SEP_med=%.4f (need .25) SEP_OK=%s"%(sep,SEP_OK))
print("=== TIER SAFETY ==="); print(tier.to_string(index=False))
print("best_tier",best_tier,"SAFE_OK",SAFE_OK)
print("USE_P3_PROTO=",USE_P3_PROTO)
decision={"auc":auc,"tpr_fpr02":tpr02,"sep_med":sep,"best_tier":best_tier,"AUC_OK":AUC_OK,"SPEC_OK":SPEC_OK,"SEP_OK":SEP_OK,"SAFE_OK":SAFE_OK,"USE_P3_PROTO":USE_P3_PROTO}
with open(WORK/"probe42_decision.json","w") as f: json.dump(decision,f,indent=2)
print("saved",WORK/"probe42_decision.json")
if USE_P3_PROTO:
    print("PASS: P3 is a new validated signal. Next is a controlled A/B only, not a stack.")
else:
    print("FAIL: P3 prototype does not beat the NB41 gates out-of-fold. Do not create a P3 submission.")


=== NB42 P3 OOF QUALITY ===
n_poison 96 n_clean 61
AUC=0.8854 (need .90) AUC_OK=False
TPR@FPR2%=0.2396 (need .60) SPEC_OK=False threshold=0.9697
SEP_med=0.7668 (need .25) SEP_OK=True
=== TIER SAFETY ===
 keep_proto_max  n_keep  clean_ratio  poison_ratio
           0.10      30     0.647984      0.049156
           0.15      34     0.735973      0.049156
           0.20      36     0.773753      0.049156
           0.25      38     0.819931      0.049156
           0.30      43     0.851334      0.102084
           0.35      43     0.851334      0.102084
           0.40      46     0.918350      0.102084
best_tier {'keep_proto_max': 0.4, 'n_keep': 46.0, 'clean_ratio': 0.9183502354616143, 'poison_ratio': 0.10208382629147097} SAFE_OK False
USE_P3_PROTO= False
saved /kaggle/working/probe42_decision.json
FAIL: P3 prototype does not beat the NB41 gates out-of-fold. Do not create a P3 submission.
